# Video → Frame extraction (Stage 1 of the pipeline)

This notebook covers the first two components of the content-verification pipeline:

1. **Video upload** — three ways to get the video into Colab (direct upload, Google Drive, or an existing path)
2. **Frame extraction agent** — ffmpeg samples 1 frame every 2 minutes and saves them to `frames_raw/`, with a timestamped `manifest.json`

It stops **before** perceptual dedup — all extracted frames stay on disk in `frames_raw/`.

**Run the cells top to bottom.** The extraction agent is idempotent — re-running it clears and regenerates the frames.

In [ ]:
# @title 1. Setup — install dependencies { display-mode: "form" }
# ffmpeg ships with Colab, so only Python deps are needed.
!pip install -q pillow

import json, math, shutil, subprocess, sys, time
from dataclasses import dataclass, asdict, field
from pathlib import Path
from datetime import datetime, timezone

from PIL import Image

# Verify ffmpeg / ffprobe are actually present
for tool in ("ffmpeg", "ffprobe"):
    r = subprocess.run([tool, "-version"], capture_output=True, text=True)
    assert r.returncode == 0, f"{tool} not found — run: !apt-get install -y ffmpeg"
    print(r.stdout.splitlines()[0])
print("\n✅ Environment ready")


In [ ]:
# @title 2. Pipeline configuration { display-mode: "form" }

@dataclass
class PipelineConfig:
    # --- frame extraction ---
    frame_interval_sec: int = 120          # 1 frame every 2 minutes
    image_format: str = "jpg"              # jpg keeps size down; use "png" for lossless
    jpeg_quality: int = 2                  # ffmpeg -q:v scale: 2 = high quality, 31 = worst
    accurate_seek: bool = False            # True = one ffmpeg seek per frame (exact timestamps, slower)
                                           # False = single-pass fps filter (fast, timestamps ≈ n*interval)
    scale_max_width: int = 1280            # downscale huge videos; 1280px is plenty for slide OCR.
                                           # set to 0 to keep original resolution

    # --- paths ---
    work_dir: str = "/content/pipeline"

    def __post_init__(self):
        self.frames_dir     = str(Path(self.work_dir) / "frames_raw")
        self.manifest_path  = str(Path(self.work_dir) / "manifest.json")

CFG = PipelineConfig()

for d in (CFG.work_dir, CFG.frames_dir):
    Path(d).mkdir(parents=True, exist_ok=True)

print(json.dumps({k: v for k, v in asdict(CFG).items()}, indent=2))


## Stage 0 — Video upload

Pick **one** of the three options below. For anything over ~200 MB, Google Drive (Option B) is far more reliable than the browser uploader.

> ⚠️ Make sure you have legitimate offline access to the video (e.g. Pluralsight's own offline feature on your account). Don't feed this pipeline content you aren't licensed to store.

In [ ]:
# @title Option A — direct browser upload (good for small files < ~200 MB)
from google.colab import files

uploaded = files.upload()  # opens a file picker
VIDEO_PATH = str(Path("/content") / next(iter(uploaded.keys())))
print("VIDEO_PATH =", VIDEO_PATH)


In [ ]:
# @title Option B — mount Google Drive (recommended for large files)
from google.colab import drive
drive.mount("/content/drive")

# ── EDIT THIS to point at your file inside Drive ──
VIDEO_PATH = "/content/drive/MyDrive/videos/course.mp4"

assert Path(VIDEO_PATH).exists(), f"Not found: {VIDEO_PATH} — fix the path above"
print("VIDEO_PATH =", VIDEO_PATH)


In [ ]:
# @title Option C — file already on the Colab disk (e.g. wget/rsync'd earlier)
VIDEO_PATH = "/content/course.mp4"   # ← edit
assert Path(VIDEO_PATH).exists(), f"Not found: {VIDEO_PATH}"
print("VIDEO_PATH =", VIDEO_PATH)


In [ ]:
# @title 3. Probe the video (duration, resolution, fps) and sanity-check

def probe_video(path: str) -> dict:
    """Return container + first-video-stream metadata via ffprobe."""
    cmd = [
        "ffprobe", "-v", "error",
        "-select_streams", "v:0",
        "-show_entries", "stream=width,height,avg_frame_rate,codec_name",
        "-show_entries", "format=duration,size,format_name",
        "-of", "json", path,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffprobe failed:\n{r.stderr}")
    meta = json.loads(r.stdout)
    stream, fmt = meta["streams"][0], meta["format"]
    num, den = (stream.get("avg_frame_rate") or "0/1").split("/")
    fps = (float(num) / float(den)) if float(den) else 0.0
    return {
        "path": path,
        "codec": stream.get("codec_name"),
        "width": stream.get("width"),
        "height": stream.get("height"),
        "fps": round(fps, 3),
        "duration_sec": float(fmt["duration"]),
        "size_mb": round(int(fmt["size"]) / 1e6, 1),
        "container": fmt.get("format_name"),
    }

VIDEO_INFO = probe_video(VIDEO_PATH)
expected_frames = math.floor(VIDEO_INFO["duration_sec"] / CFG.frame_interval_sec) + 1

print(json.dumps(VIDEO_INFO, indent=2))
print(f"\nDuration: {VIDEO_INFO['duration_sec']/60:.1f} min "
      f"→ expecting ~{expected_frames} frames at 1 per {CFG.frame_interval_sec}s")

if VIDEO_INFO["duration_sec"] < CFG.frame_interval_sec:
    print("⚠️ Video is shorter than the sampling interval — you'll get a single frame.")


## Stage 1 — Frame extraction agent

Two extraction modes, controlled by `CFG.accurate_seek`:

- **Fast (default)** — one ffmpeg pass with `fps=1/120`. Frame *n* lands at *t ≈ (n−1)·120s*. For lecture content this is exact enough, and it's a single decode of the file.
- **Accurate** — one `ffmpeg -ss <t>` seek per timestamp. Frame-exact, but launches a process per frame (still fast because `-ss` before `-i` uses keyframe seeking).

Every extracted frame gets a manifest entry carrying its timestamp forward — the final coverage report depends on these.

In [ ]:
# @title 4. Frame extraction agent

class FrameExtractionAgent:
    """Extracts 1 frame every `frame_interval_sec` seconds and builds a timestamped manifest."""

    def __init__(self, cfg: PipelineConfig, video_info: dict):
        self.cfg, self.info = cfg, video_info

    # ---------- public ----------
    def run(self) -> dict:
        out_dir = Path(self.cfg.frames_dir)
        for old in out_dir.glob(f"*.{self.cfg.image_format}"):
            old.unlink()                                # idempotent re-runs

        t0 = time.time()
        if self.cfg.accurate_seek:
            frames = self._extract_accurate(out_dir)
        else:
            frames = self._extract_fast(out_dir)
        elapsed = time.time() - t0

        manifest = {
            "video": self.info,
            "created_utc": datetime.now(timezone.utc).isoformat(),
            "config": asdict(self.cfg),
            "stages": {"extraction": {"mode": "accurate" if self.cfg.accurate_seek else "fast",
                                      "elapsed_sec": round(elapsed, 1),
                                      "frame_count": len(frames)}},
            "frames": frames,
        }
        Path(self.cfg.manifest_path).write_text(json.dumps(manifest, indent=2))
        print(f"✅ Extracted {len(frames)} frames in {elapsed:.1f}s → {out_dir}")
        return manifest

    # ---------- internals ----------
    def _scale_filter(self) -> str:
        w = self.cfg.scale_max_width
        return f",scale='min({w},iw)':-2" if w else ""

    def _frame_record(self, path: Path, index: int, ts: float) -> dict:
        return {
            "frame_id": path.stem,
            "index": index,
            "timestamp_sec": round(ts, 2),
            "timestamp_hms": time.strftime("%H:%M:%S", time.gmtime(ts)),
            "path": str(path),
            "status": "extracted",
        }

    def _extract_fast(self, out_dir: Path) -> list:
        pattern = str(out_dir / f"frame_%04d.{self.cfg.image_format}")
        dur = self.info["duration_sec"]
        # tpad clones the last real frame out to the container duration: screen recordings
        # often stop emitting video frames before the audio/container ends, which would
        # otherwise silently truncate sampling (e.g. 7 frames from a 14.5-min video).
        vf = (f"tpad=stop_mode=clone:stop_duration={dur},"
              f"fps=1/{self.cfg.frame_interval_sec}{self._scale_filter()}")
        cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
               "-i", self.info["path"], "-vf", vf,
               "-t", f"{dur + self.cfg.frame_interval_sec / 2:.3f}",
               "-q:v", str(self.cfg.jpeg_quality), "-fps_mode", "vfr", pattern]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"ffmpeg failed:\n{r.stderr}")
        frames = []
        for i, p in enumerate(sorted(out_dir.glob(f"frame_*.{self.cfg.image_format}"))):
            frames.append(self._frame_record(p, i, i * self.cfg.frame_interval_sec))
        return frames

    def _extract_accurate(self, out_dir: Path) -> list:
        frames, ts, i = [], 0.0, 0
        while ts < self.info["duration_sec"]:
            p = out_dir / f"frame_{i:04d}.{self.cfg.image_format}"
            vf = f"scale='min({self.cfg.scale_max_width},iw)':-2" if self.cfg.scale_max_width else "null"
            cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
                   "-ss", f"{ts:.3f}", "-i", self.info["path"],
                   "-frames:v", "1", "-vf", vf,
                   "-q:v", str(self.cfg.jpeg_quality), str(p)]
            r = subprocess.run(cmd, capture_output=True, text=True)
            if r.returncode != 0:
                raise RuntimeError(f"ffmpeg failed at t={ts}:\n{r.stderr}")
            if not p.exists() and frames:
                # timestamp is past the last real video frame — clone the previous one
                shutil.copy(frames[-1]["path"], p)
            if p.exists():
                frames.append(self._frame_record(p, i, ts))
            ts += self.cfg.frame_interval_sec
            i += 1
            if i % 10 == 0:
                print(f"  … {i} frames ({ts/60:.0f} min in)")
        return frames


extractor = FrameExtractionAgent(CFG, VIDEO_INFO)
manifest = extractor.run()

expected = math.floor(VIDEO_INFO["duration_sec"] / CFG.frame_interval_sec) + 1
got = len(manifest["frames"])
if got < expected:
    print(f"⚠️ Expected ~{expected} frames for a "
          f"{VIDEO_INFO['duration_sec']/60:.1f}-min video but got {got}. "
          f"The video stream may end before the container does.")
else:
    print(f"Frame count matches expectation ({got}/{expected}).")

for f in manifest["frames"]:      # full listing
    print(f'{f["frame_id"]}  @ {f["timestamp_hms"]}')


In [ ]:
# @title 5. Visual check — contact sheet of extracted frames
import matplotlib.pyplot as plt

frames = manifest["frames"]
cols = 4
rows = math.ceil(len(frames) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, 3 * rows))
axes = axes.flat if hasattr(axes, "flat") else [axes]
for ax, rec in zip(axes, frames):
    ax.imshow(Image.open(rec["path"]))
    ax.set_title(f'{rec["frame_id"]} @ {rec["timestamp_hms"]}', fontsize=9)
    ax.axis("off")
for ax in list(axes)[len(frames):]:
    ax.axis("off")
plt.suptitle(f"All extracted frames ({len(frames)})")
plt.tight_layout(); plt.show()


In [ ]:
# @title 6. Zip frames + manifest (input for the perceptual dedup stage)
archive = Path(CFG.work_dir) / "stage1_output"
if archive.with_suffix(".zip").exists():
    archive.with_suffix(".zip").unlink()

staging = Path(CFG.work_dir) / "_staging"
if staging.exists(): shutil.rmtree(staging)
staging.mkdir()
shutil.copytree(CFG.frames_dir, staging / "frames_raw")
shutil.copy(CFG.manifest_path, staging / "manifest.json")
zip_path = shutil.make_archive(str(archive), "zip", staging)
shutil.rmtree(staging)
print("Created:", zip_path, f"({Path(zip_path).stat().st_size/1e6:.1f} MB)")

# Option 1 — browser download:
# from google.colab import files; files.download(zip_path)

# Option 2 — copy to Drive (if mounted):
# shutil.copy(zip_path, "/content/drive/MyDrive/pipeline/stage1_output.zip")


## Manifest after this stage

```json
{
  "video": {"path": "...", "duration_sec": 3720.5, "width": 1280, ...},
  "stages": {"extraction": {"mode": "fast", "frame_count": 32, "elapsed_sec": 4.1}},
  "frames": [
    {"frame_id": "frame_0001", "index": 0, "timestamp_sec": 0,
     "timestamp_hms": "00:00:00", "path": ".../frames_raw/frame_0001.jpg", "status": "extracted"},
    ...
  ]
}
```

Every frame is on disk in `frames_raw/`, untouched. The **perceptual pre-dedup agent** runs next, reading this manifest and moving near-identical frames aside before any Qwen inference.